<a href="https://colab.research.google.com/github/ha0-922/VLA/blob/main/LeRobot_Diffusion_Policy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 설치
!pip install lerobot gym-pusht -q

print("설치 완료!")

In [ ]:
# 2. pretrained 모델 로드
from lerobot.common.policies.diffusion.modeling_diffusion import DiffusionPolicy
from huggingface_hub import hf_hub_download
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 디바이스:", device)

policy = DiffusionPolicy.from_pretrained("lerobot/diffusion_pusht")
policy = policy.to(device)
policy.eval()

print("✅ 모델 로드 완료!")

In [ ]:
# 구조 확인
import lerobot
print(lerobot.__version__)
print(lerobot.__file__)

!find /usr/local/lib/python3.11/dist-packages/lerobot -name "*.py" | head -20

In [ ]:
!find /usr/local/lib/python3.12/dist-packages/lerobot -type d

In [ ]:
import torch
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 디바이스:", device)

policy = DiffusionPolicy.from_pretrained("lerobot/diffusion_pusht")
policy = policy.to(device)
policy.eval()

print("✅ 모델 로드 완료!")

In [ ]:
!pip install "pymunk==6.6.0" -q

In [ ]:
# 셀 1: 모델 다시 로드
import torch
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = DiffusionPolicy.from_pretrained("lerobot/diffusion_pusht")
policy = policy.to(device)
policy.eval()
print("✅ 모델 로드 완료!")

In [ ]:
import gymnasium as gym
import gym_pusht

env = gym.make("gym_pusht/PushT-v0", render_mode="rgb_array")
obs, _ = env.reset()

print("obs 타입:", type(obs))
print("obs 내용:", obs)

In [ ]:
import gymnasium as gym
import gym_pusht
import numpy as np
import torch
import imageio
from IPython.display import Video

# pixels 포함 버전으로 환경 생성
env = gym.make("gym_pusht/PushT-v0",
               render_mode="rgb_array",
               obs_type="pixels_agent_pos")  # 이미지 + 위치 같이 받기
obs, _ = env.reset()

print("obs 타입:", type(obs))
print("obs 키:", obs.keys() if hasattr(obs, 'keys') else obs.shape)

In [ ]:
%cd /content/octo

import jax
import numpy as np
print("jax:", jax.__version__)
print("numpy:", np.__version__)
print("devices:", jax.devices())

In [ ]:
import gymnasium as gym
import gym_pusht

env = gym.make("gym_pusht/PushT-v0",
               render_mode="rgb_array",
               obs_type="pixels_agent_pos")
obs, _ = env.reset()

print("obs 타입:", type(obs))
print("obs 키:", obs.keys() if hasattr(obs, 'keys') else obs.shape)

In [ ]:
frames = []
policy.reset()

for step in range(200):
    image = torch.from_numpy(obs["pixels"]).float() / 255.0
    image = image.permute(2, 0, 1).unsqueeze(0).to(device)
    state = torch.from_numpy(obs["agent_pos"]).float().unsqueeze(0).to(device)

    observation = {
        "observation.image": image,
        "observation.state": state,
    }

    with torch.no_grad():
        action = policy.select_action(observation)

    action_np = action.squeeze(0).cpu().numpy()
    obs, reward, terminated, truncated, info = env.step(action_np)
    frames.append(env.render())

    if terminated or truncated:
        obs, _ = env.reset()
        policy.reset()

env.close()

imageio.mimsave("pusht_demo.mp4", frames, fps=15)
print(f"✅ 영상 저장 완료! ({len(frames)} 프레임)")
Video("pusht_demo.mp4", width=400)

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open("pusht_demo.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=400 controls><source src="{data_url}" type="video/mp4"></video>')